In [1]:
# ============================================================
# ERIP - Loan Origination Bronze Ingestion
# Notebook: nb_ingest_loan_origination
# Purpose:
# 1. Read Loan Origination source file
# 2. Validate data contract
# 3. Run data quality checks
# 4. Write Bronze Delta table
# 5. Log ingestion metadata and DQ results
# ============================================================

# ====================================================
# SECTION 1
# Pipeline Initialization
# ====================================================

from pyspark.sql.functions import *
from pyspark.sql.types import *
from datetime import datetime

source_system = "Loan Origination"
source_file_path = "Files/01_Bronze/loan_origination/loan_origination.csv"
target_table = "bronze_loan_origination"
pipeline_name = "nb_ingest_loan_origination"

run_start_time = datetime.now()

print("ERIP Loan Origination ingestion started")
print(f"Source system: {source_system}")
print(f"Source file: {source_file_path}")
print(f"Target table: {target_table}")

StatementMeta(, e29f7247-b7f6-4caa-8a82-0ccc63713227, 3, Finished, Available, Finished, False)

ERIP Loan Origination ingestion started
Source system: Loan Origination
Source file: Files/01_Bronze/loan_origination/loan_origination.csv
Target table: bronze_loan_origination


In [2]:
# ====================================================
# SECTION 2
# Read source CSV
# ====================================================

loan_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(source_file_path)
)

display(loan_df.limit(10))

print(f"Rows read: {loan_df.count()}")
print(f"Columns read: {len(loan_df.columns)}")

StatementMeta(, e29f7247-b7f6-4caa-8a82-0ccc63713227, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 17808e93-0077-4e3c-80a0-6d6f12e5c9ae)

Rows read: 5000
Columns read: 23


In [5]:
# ============================================================
# SECTION 3 - DATA CONTRACT VALIDATION
# ============================================================

required_columns = [
    "loan_id",
    "facility_id",
    "customer_id",
    "customer_group_id",
    "product_type",
    "facility_status",
    "origination_date",
    "maturity_date",
    "approved_limit",
    "outstanding_balance",
    "undrawn_amount",
    "currency",
    "interest_rate_pct",
    "repayment_schedule",
    "collateral_id",
    "collateral_type",
    "collateral_value",
    "loan_to_value_pct",
    "ifrs9_stage",
    "risk_weight",
    "risk_weighted_assets",
    "source_system",
    "extract_date"
]

missing_columns = list(
    set(required_columns) -
    set(loan_df.columns)
)

if len(missing_columns) == 0:
    print("✓ Data Contract Validation Passed")
else:
    print("✗ Missing Columns:")
    print(missing_columns)

StatementMeta(, e29f7247-b7f6-4caa-8a82-0ccc63713227, 7, Finished, Available, Finished, False)

✓ Data Contract Validation Passed


In [6]:
# ============================================================
# SECTION 4 - THREE-LAYER DATA CONTRACT VALIDATION
# Layer 1: Schema Rules
# Layer 2: Business Rules
# Layer 3: Regulatory / Banking Rules
# ============================================================

validation_results = []

def add_validation_result(layer, rule_id, rule_name, failed_count):
    status = "PASS" if failed_count == 0 else "FAIL"
    validation_results.append({
        "pipeline_name": pipeline_name,
        "source_system": source_system,
        "target_table": target_table,
        "validation_layer": layer,
        "rule_id": rule_id,
        "rule_name": rule_name,
        "failed_count": int(failed_count),
        "status": status,
        "validation_timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    })

# ----------------------------
# Layer 1: Schema Rules
# ----------------------------
add_validation_result(
    "Schema",
    "SCHEMA_001",
    "All required columns must be present",
    len(missing_columns)
)

# ----------------------------
# Layer 2: Business Rules
# ----------------------------
total_rows = loan_df.count()

duplicate_loan_ids = total_rows - loan_df.select("loan_id").distinct().count()
null_loan_ids = loan_df.filter(col("loan_id").isNull()).count()
null_customer_ids = loan_df.filter(col("customer_id").isNull()).count()
invalid_approved_limit = loan_df.filter(col("approved_limit") <= 0).count()
invalid_outstanding_balance = loan_df.filter(col("outstanding_balance") < 0).count()
outstanding_exceeds_limit = loan_df.filter(col("outstanding_balance") > col("approved_limit")).count()
invalid_interest_rate = loan_df.filter(col("interest_rate_pct") <= 0).count()
invalid_dates = loan_df.filter(to_date(col("maturity_date")) <= to_date(col("origination_date"))).count()
invalid_facility_status = loan_df.filter(
    ~col("facility_status").isin("Active", "Restructured", "Matured", "Default")
).count()

add_validation_result("Business", "BUS_001", "Loan ID must be unique", duplicate_loan_ids)
add_validation_result("Business", "BUS_002", "Loan ID must not be null", null_loan_ids)
add_validation_result("Business", "BUS_003", "Customer ID must not be null", null_customer_ids)
add_validation_result("Business", "BUS_004", "Approved limit must be greater than zero", invalid_approved_limit)
add_validation_result("Business", "BUS_005", "Outstanding balance must not be negative", invalid_outstanding_balance)
add_validation_result("Business", "BUS_006", "Outstanding balance must not exceed approved limit", outstanding_exceeds_limit)
add_validation_result("Business", "BUS_007", "Interest rate must be greater than zero", invalid_interest_rate)
add_validation_result("Business", "BUS_008", "Maturity date must be after origination date", invalid_dates)
add_validation_result("Business", "BUS_009", "Facility status must be valid", invalid_facility_status)

# ----------------------------
# Layer 3: Regulatory / Banking Rules
# ----------------------------
invalid_ifrs9_stage = loan_df.filter(
    ~col("ifrs9_stage").isin("Stage 1", "Stage 2", "Stage 3")
).count()

invalid_risk_weight = loan_df.filter(col("risk_weight") < 0).count()
invalid_rwa = loan_df.filter(col("risk_weighted_assets") < 0).count()

secured_missing_collateral_id = loan_df.filter(
    (col("collateral_type") != "Unsecured") &
    ((col("collateral_id").isNull()) | (trim(col("collateral_id")) == ""))
).count()

secured_invalid_collateral_value = loan_df.filter(
    (col("collateral_type") != "Unsecured") &
    (col("collateral_value") <= 0)
).count()

invalid_currency = loan_df.filter(~col("currency").isin("EUR")).count()

invalid_ltv = loan_df.filter(
    (col("collateral_type") != "Unsecured") &
    ((col("loan_to_value_pct").isNull()) | (col("loan_to_value_pct") < 0))
).count()

add_validation_result("Regulatory", "REG_001", "IFRS 9 stage must be valid", invalid_ifrs9_stage)
add_validation_result("Regulatory", "REG_002", "Risk weight must not be negative", invalid_risk_weight)
add_validation_result("Regulatory", "REG_003", "Risk weighted assets must not be negative", invalid_rwa)
add_validation_result("Regulatory", "REG_004", "Secured loans must have collateral ID", secured_missing_collateral_id)
add_validation_result("Regulatory", "REG_005", "Secured loans must have collateral value greater than zero", secured_invalid_collateral_value)
add_validation_result("Regulatory", "REG_006", "Currency must be valid", invalid_currency)
add_validation_result("Regulatory", "REG_007", "Loan-to-value must be populated for secured loans", invalid_ltv)

validation_df = spark.createDataFrame(validation_results)

display(validation_df)

failed_validations = validation_df.filter(col("status") == "FAIL").count()

if failed_validations > 0:
    raise Exception(f"Data Contract Failed: {failed_validations} validation rule(s) failed.")
else:
    print("✓ Loan Origination Three-layer Validation Passed")

StatementMeta(, e29f7247-b7f6-4caa-8a82-0ccc63713227, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, dc7d9358-3a25-4c19-9592-edc60296aee7)

✓ Loan Origination Three-layer Validation Passed


In [7]:
# ============================================================
# SECTION 4 - VALIDATION FRAMEWORK
# Data Contract + Data Quality + Business + Regulatory Checks
# ============================================================

# Add ingestion audit columns to source data
loan_bronze_df = (
    loan_df
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("pipeline_name", lit(pipeline_name))
    .withColumn("bronze_load_date", current_date())
)

# Write Bronze Delta table
loan_bronze_df.write.mode("overwrite").format("delta").saveAsTable("bronze_loan_origination")

# Write validation results table
validation_df.write.mode("append").format("delta").saveAsTable("dq_validation_results")

print("✓ Bronze Delta table created: bronze_loan_origination")
print("✓ DQ validation results written: dq_validation_results")
print(f"Rows written to Bronze: {loan_bronze_df.count()}")

StatementMeta(, e29f7247-b7f6-4caa-8a82-0ccc63713227, 9, Finished, Available, Finished, False)

✓ Bronze Delta table created: bronze_loan_origination
✓ DQ validation results written: dq_validation_results
Rows written to Bronze: 5000


In [8]:
# ============================================================
# SECTION 5 - DATA QUALITY SUMMARY
# ============================================================

total_validation_rules = validation_df.count()
passed_validation_rules = validation_df.filter(col("status") == "PASS").count()
failed_validation_rules = validation_df.filter(col("status") == "FAIL").count()

dq_score = (passed_validation_rules / total_validation_rules) * 100

print("Data Quality Summary")
print("--------------------")
print(f"Total validation rules: {total_validation_rules}")
print(f"Passed validation rules: {passed_validation_rules}")
print(f"Failed validation rules: {failed_validation_rules}")
print(f"Data Quality Score: {dq_score}%")

StatementMeta(, e29f7247-b7f6-4caa-8a82-0ccc63713227, 10, Finished, Available, Finished, False)

Data Quality Summary
--------------------
Total validation rules: 17
Passed validation rules: 17
Failed validation rules: 0
Data Quality Score: 100.0%


In [9]:
# ============================================================
# SECTION 6 - METADATA LOGGING
# ============================================================

from datetime import datetime

run_end_time = datetime.now()
execution_time_seconds = (run_end_time - run_start_time).total_seconds()

metadata = [{
    "pipeline_name": pipeline_name,
    "source_system": source_system,
    "target_table": target_table,
    "rows_processed": loan_bronze_df.count(),
    "validation_rules": total_validation_rules,
    "dq_score": dq_score,
    "status": "SUCCESS",
    "run_start_time": run_start_time.strftime("%Y-%m-%d %H:%M:%S"),
    "run_end_time": run_end_time.strftime("%Y-%m-%d %H:%M:%S"),
    "execution_time_seconds": execution_time_seconds
}]

metadata_df = spark.createDataFrame(metadata)

metadata_df.write \
    .mode("append") \
    .format("delta") \
    .saveAsTable("metadata_ingestion_log")

display(metadata_df)

print("✓ Metadata successfully written")

StatementMeta(, e29f7247-b7f6-4caa-8a82-0ccc63713227, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8bcd570e-31ec-40ae-96fc-17219cc0f54b)

✓ Metadata successfully written
